# 19. Remove Nth Node From End of List

[Problem](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) · difficulty: medium · topics: linked-list, two-pointers

This notebook visually traces the two-pointer sliding window algorithm (`SolutionTwoPointers`).
It diagrams the pointer movements step-by-step, showing how an $n$-step lead
locates the target node in a single pass, how the unlinking bypass works,
and how edge cases (head and single-node deletion) are handled.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0019-remove-nth-node-from-end-of-list'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
[s.__name__ for s in solutions]

## The invariant: fixed-gap sliding window

Because the list is singly linked, we cannot walk backward from the end.
Instead, we create a **rigid window of width $n$** between two pointers:

```
1. Create gap of n=2:
   [1] ──> [2] ──> [3] ──> [4] ──> [5] ──> None
    │<─── gap = 2 ───>│
 slow              h

2. Slide the entire window forward until fast reaches None:
   [1] ──> [2] ──> [3] ──> [4] ──> [5] ──> None
                    │               │<─── gap = 2 ───>│
                   prev          slow              h
                              (target node!)

3. Unlink the target node by re-pointing prev.next:
   [1] ──> [2] ──> [3] ──────────────────> [5] ──> None
                    │                       ▲
                    │     ┌─── [4] ───┐     │
                    └────>│ (bypassed)├─────┘
```

When the front of the window `h` falls off the end of the list at `None`,
the back of the window `slow` is resting on the $n$-th node from the end.


In [ ]:
# Visual renderer for linked list and pointers

def render_diagram(vals, fast_idx, slow_idx, prev_idx, title="", note=""):
    node_strs = [f"[{v}]" for v in vals] + ["None"]
    line = " ──> ".join(node_strs)
    
    # Calculate center position of each node on screen
    positions = []
    pos = 0
    for s in node_strs:
        positions.append(pos + len(s) // 2)
        pos += len(s) + 5  # len(" ──> ") == 5
    
    max_len = positions[-1] + 10
    carets = [" "] * max_len
    labels = [" "] * max_len
    
    # Map nodes to active pointers
    ptrs = {}
    if prev_idx is not None:
        ptrs.setdefault(prev_idx, []).append("prev")
    if slow_idx is not None:
        ptrs.setdefault(slow_idx, []).append("slow")
    if fast_idx is not None:
        ptrs.setdefault(fast_idx, []).append("fast")
        
    for idx in sorted(ptrs.keys()):
        p = positions[idx]
        carets[p] = "▲"
        tag = ", ".join(ptrs[idx])
        start = max(0, p - len(tag) // 2)
        for i, ch in enumerate(tag):
            if start + i < max_len:
                labels[start + i] = ch
                
    if title:
        print(title)
    print("  " + line)
    print("  " + "".join(carets).rstrip())
    print("  " + "".join(labels).rstrip())
    if note:
        print(f"  {note}")
    print()


## Step-by-step trace: Example 1

Input: `head = [1, 2, 3, 4, 5]`, `n = 2` (remove node with value `4`).

In [ ]:
vals = [1, 2, 3, 4, 5]
n = 2

print("=" * 65)
print(f"STARTING TRACE: vals={vals}, n={n}")
print("=" * 65)

# Step 0: Initial state
fast_idx = 0
slow_idx = 0
prev_idx = None
render_diagram(vals, fast_idx, slow_idx, prev_idx, "[Step 0: Initial Position]", "slow and h both start at head; prev is None")

# Phase 1: Advance fast by n steps
print("--- Phase 1: Advance fast by n=2 steps to establish window gap ---")
for step in range(1, n + 1):
    fast_idx += 1
    render_diagram(vals, fast_idx, slow_idx, prev_idx, f"[Lead Phase: step {step} of {n}]", f"fast advanced to [{vals[fast_idx]}], gap = {fast_idx - slow_idx}")

# Phase 2: Joint lockstep advance
print("--- Phase 2: Advance fast, slow, and prev in lockstep until fast reaches None ---")
step_count = 0
while fast_idx < len(vals):
    step_count += 1
    fast_idx += 1
    prev_idx = slow_idx
    slow_idx += 1
    h_label = f"[{vals[fast_idx]}]" if fast_idx < len(vals) else "None"
    render_diagram(vals, fast_idx, slow_idx, prev_idx, f"[Lockstep step {step_count}]", f"prev=[{vals[prev_idx]}], slow=[{vals[slow_idx]}], h={h_label}")

# Phase 3: Unlink
print("--- Phase 3: Unlink slow node (value 4) ---")
print("Action: prev.next = prev.next.next")
print("  [1] ──> [2] ──> [3] ──────────────────> [5] ──> None")
print("                   │                       ▲")
print("                   │     ┌─── [4] ───┐     │")
print("                   └────>│ (bypassed)├─────┘")
print("                         └───────────┘")
print("Final list: [1, 2, 3, 5]")

## Edge cases: head, tail, and single-node list

### Head removal (`n == sz`)
Consider `vals = [1, 2]`, `n = 2`. The target to remove is `[1]`.

Because $n = sz$, the lead phase advances `h` all the way to `None` immediately.
The lockstep `while` loop never executes, leaving `prev = None`:

In [ ]:
vals_head = [1, 2]
n_head = 2

render_diagram(vals_head, 0, 0, None, "[Initial State]", "slow and h at [1]")
render_diagram(vals_head, 2, 0, None, "[After Phase 1: fast advanced n=2 steps]", "fast reaches None immediately! Lockstep phase is skipped.")

print("Branch check: prev is None and slow is not None")
print("Action: head = slow.next (new head becomes [2])")
print("  [1] ──> [2] ──> None")
print("   │       ▲")
print("   └──X    └── new head")
print("Result: [2]")

### Single-node list (`sz = 1, n = 1`)
Consider `vals = [1]`, `n = 1`. The only node is removed, leaving an empty list:

In [ ]:
vals_single = [1]
render_diagram(vals_single, 0, 0, None, "[Initial State]", "slow and h at [1]")
render_diagram(vals_single, 1, 0, None, "[After Phase 1: fast advanced n=1 step]", "fast reaches None. prev is None, slow.next is None.")
print("Action: head = slow.next -> None")
print("Result: [] (empty list)")

### Tail removal (`n = 1`)
Consider `vals = [1, 2]`, `n = 1`. The target to remove is the last element `[2]`:

In [ ]:
vals_tail = [1, 2]
render_diagram(vals_tail, 0, 0, None, "[Initial State]")
render_diagram(vals_tail, 1, 0, None, "[Phase 1: fast advanced n=1 step]", "gap of 1 established")
render_diagram(vals_tail, 2, 1, 0, "[Phase 2: lockstep step 1]", "fast reaches None. slow at [2], prev at [1]")
print("Action: prev.next = None")
print("  [1] ──> None (node [2] disconnected)")
print("Result: [1]")

## Verification across all test cases

We run `SolutionTwoPointers` against all 12 cases declared in `test_remove_nth_node_from_end_of_list.py`:

In [ ]:
import copy, sys
from lc.harness import load_module

# Ensure clean unmutated test cases even if cell is re-executed in an active kernel
test_mod_name = f"lc_problem_{PROBLEM.name.replace('-', '_')}_test_remove_nth_node_from_end_of_list"
sys.modules.pop(test_mod_name, None)
test_mod = load_module(PROBLEM / 'test_remove_nth_node_from_end_of_list.py')
sol = solutions[0]()

passed = 0
for (head_arg, n_arg), expected in test_mod.CASES:
    # Linked lists are mutated in place: deepcopy to preserve cases across iterations
    head_copy = copy.deepcopy(head_arg)
    actual = sol.removeNthFromEnd(head_copy, n_arg)
    assert test_mod.NORMALIZE(actual) == test_mod.NORMALIZE(expected), (
        f"Failed on input {head_arg}, n={n_arg}: got {test_mod.NORMALIZE(actual)}, expected {test_mod.NORMALIZE(expected)}"
    )
    passed += 1

print(f"All {passed} test cases passed successfully!")


## Summary

| Stage | Action | Invariant / Property |
|---|---|---|
| **Phase 1: Lead Setup** | Advance `h` by $n$ steps | Gap of distance $n$ established between `h` and `slow` |
| **Phase 2: Lockstep Slide** | Step `h`, `slow`, `prev` together | Gap maintained until `h` hits `None` at position $sz + 1$ |
| **Termination** | Stop when `h is None` | `slow` is guaranteed to be at the $n$-th node from end |
| **Phase 3: Unlink** | Re-link `prev.next` or update `head` | If `prev is None`, target is head; else bypass `slow` |

### Takeaways

- **Sliding Window on a List**: An $n$-step lead acts like a physical caliper sliding along the linked list; when the caliper's front edge hits `None`, its rear edge is pinned to the target.
- **Single Pass $O(sz)$**: Avoids counting the list length or storing nodes in auxiliary arrays.
- **$O(1)$ Auxiliary Space**: Uses only three pointer references (`h`, `slow`, `prev`).
